# Does rank-space indexing really improve residual mean reversion?

A replication check on **Li & Papanicolaou, *Statistical Arbitrage in Rank Space*,
arXiv:2410.06568v2 (rev. 29 Jun 2026)**.

Every claim about the paper below is tied to a line number in the text extracted
via `pdftotext -layout` from `https://arxiv.org/pdf/2410.06568`. Regenerate with:

```
curl -sL https://arxiv.org/pdf/2410.06568 -o rankspace.pdf
pdftotext -layout rankspace.pdf rankspace.txt
```

Then `sed -n '<line>p' rankspace.txt` reproduces any quote.

## The paper makes two separable claims

> *"This performance gain is driven by more robust market representations and
> enhanced mean-reverting properties of residual returns in rank space"* — **abstract, L22-23**

1. **Structural**: rank space has a more concentrated factor structure.
2. **Mean reversion**: rank-space residuals revert faster.

They are tested separately here. **Neither test concerns tradeability** — the
paper reports the strategy dies at 5bp (L691), so tradeability is not in dispute.

## The confound

Rank-slot returns are returns on **order statistics** of capitalisation. When two
names swap adjacent ranks, the capitalisation *at rank k* moves less than either
name did, because the ordering re-sorts. Order statistics are less volatile and
more mean-reverting than the underlying **by construction**.

The authors are not naive about this — **Appendix B (L1183+)** derives a
hybrid-Atlas model with local times, and **L1350** invokes Banner–Ghomrasni's
lemma that local times from triple-or-higher collisions vanish. Their return
definition is *motivated by* that model (L186).

So the question is not *"does rank space mean-revert?"* — it must — but
**"does it mean-revert more than sorting alone would produce?"** That requires a
null in which no cross-sectional signal exists.

## 1. The paper's construction, quoted

**Rank return, eq (3.5), L179-181:**

> r̃(k),t := (c(k),t − c(k),t−1) / c(k),t−1

where *"c(k),t is the capitalization of the stock occupying the k-th rank in
descending order at day t"* (L183-184).

**Universe and factor model, Appendix A.2, L733-741:**

> *"we re-calibrate the investment universe by selecting stocks that (i) rank
> among the top 500 in capitalization as of day t"* — **L734**
>
> *"perform principal component analysis (PCA) on the selected returns using a
> 252-day lookback window"* — **L737-738**
>
> *"we retain the top five eigenvectors (associated with the five largest
> eigenvalues) in name space, and the top eigenvector in rank space"* — **L738-740**
>
> *"Factor loadings βt are estimated using a 60-day lookback window"* — **L740**

**K = 5 in name space, K = 1 in rank space.** This asymmetry matters: removing
five factors cleans name-space residuals more thoroughly, which works *against*
a rank-space advantage. It makes their setup conservative, and it is the single
thing my first attempt got wrong.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import replicate as rp

print("Replication constants, matched to Appendix A.2:")
print(f"  TOP_N    = {rp.TOP_N}     (L734: 'top 500 in capitalization')")
print(f"  PCA_WIN  = {rp.PCA_WIN}     (L737: '252-day lookback window')")
print(f"  K_NAME   = {rp.K_NAME}       (L738: 'top five eigenvectors ... in name space')")
print(f"  K_RANK   = {rp.K_RANK}       (L739: 'the top eigenvector in rank space')")
print(f"  BETA_WIN = {rp.BETA_WIN}      (L740: 'a 60-day lookback window')")
print(f"  REBAL    = {rp.REBAL}      (deviation: they recalibrate daily; see Limitations)")

Replication constants, matched to Appendix A.2:
  TOP_N    = 500     (L734: 'top 500 in capitalization')
  PCA_WIN  = 252     (L737: '252-day lookback window')
  K_NAME   = 5       (L738: 'top five eigenvectors ... in name space')
  K_RANK   = 1       (L739: 'the top eigenvector in rank space')
  BETA_WIN = 60      (L740: 'a 60-day lookback window')
  REBAL    = 21      (deviation: they recalibrate daily; see Limitations)


## 2. Data

985 US names (S&P 500 + 400 + 600 constituents) with ≥98% price history from
2006-05, capitalisation = adjusted price × **current** shares outstanding. Top
500 by cap selected every 21 days.

**Known bias, stated up front:** the universe is currently-listed names with
static share counts. Names that fell in capitalisation and delisted are absent,
so the losing tail is truncated and each rank slot is backfilled from below.
**This manufactures mean reversion exactly where the paper claims to find it** —
i.e. it biases *toward* the paper's hypothesis. A null result is therefore
informative; a positive one would not have been.

In [2]:
px = pd.read_parquet("cache/prices_big.parquet")
px = px[px.index >= "2006-01-01"].dropna(axis=1, thresh=int(len(px)*0.98)).ffill().dropna()
sh = pd.read_parquet("cache/shares_big.parquet")["shares"]
cols = [c for c in px.columns if c in sh.index]
cap = px[cols] * sh.reindex(cols)
print(f"{len(cols)} names | {len(cap):,} trading days | "
      f"{cap.index.min():%Y-%m} to {cap.index.max():%Y-%m}")
print(f"paper's backtest span (L731): January 2006 to December 2022")

985 names | 5,109 trading days | 2006-05 to 2026-09
paper's backtest span (L731): January 2006 to December 2022


## 3. The null

Identical pipeline, driven by **simulated** capitalisations:

- one common factor (share 0.40) plus idiosyncratic noise
- per-name volatility and starting capitalisations matched to the real data
- **zero mean reversion and zero cross-sectional predictability by construction**

The order-statistic effect is present in both arms because both are sorted the
same way. Only genuine structure can separate them.

In [3]:
import inspect
print(inspect.getsource(rp.simulate_caps))

def simulate_caps(cap: pd.DataFrame, rng, factor_share: float) -> pd.DataFrame:
    """Simulated caps: common factor + idiosyncratic, matched per-name vol and
    starting caps, NO mean reversion and NO cross-sectional predictability."""
    r = (cap / cap.shift(1) - 1).iloc[1:]
    sig = r.std().to_numpy()
    T, N = len(cap) - 1, cap.shape[1]
    f = rng.standard_normal((T, 1))
    e = rng.standard_normal((T, N))
    sh = (np.sqrt(factor_share) * f + np.sqrt(1 - factor_share) * e) * sig
    logc = np.log(cap.iloc[0].to_numpy()) + np.cumsum(sh, axis=0)
    return pd.DataFrame(np.vstack([cap.iloc[0].to_numpy(), np.exp(logc)]),
                        index=cap.index, columns=cap.columns)



## 4. Result 1 — mean reversion

Compare each space to **its own** null. (My first attempt used a difference
statistic `ar1(name) − ar1(rank)`, which penalises the real data for having
genuine name-space reversal that the null lacks by construction. That was a
mistake; per-space comparison is the right framing.)

In [4]:
import json
r = json.load(open("replication_results.json"))
tbl = pd.DataFrame({
    "observed":     [r["ar1_name"], r["ar1_rank"]],
    "null median":  [r["null_name_median"], r["null_rank_median"]],
    "null 5th pct": [np.nan, r["null_rank_p5"]],
}, index=["name-space AR(1), K=5", "rank-space AR(1), K=1"])
tbl["excess vs null"] = tbl["observed"] - tbl["null median"]
print(tbl.round(4).to_string())
print(f"\nfraction of null rank-space draws at least as negative as observed: "
      f"{r['frac_null_le_obs']:.1%}")
print(f"null draws: {r['n_null']}, factor share {r['factor_share']}")

                       observed  null median  null 5th pct  excess vs null
name-space AR(1), K=5   -0.0113      -0.0013           NaN         -0.0101
rank-space AR(1), K=1   -0.1587      -0.1588       -0.1688          0.0001

fraction of null rank-space draws at least as negative as observed: 50.0%
null draws: 30, factor share 0.4


**Rank-space AR(1) is −0.1587 against a null median of −0.1588** — a discrepancy
in the fourth decimal, sitting at the **50th percentile** of the null.

**Name-space AR(1) is −0.0113 against a null median of −0.0013** — roughly 9x the
null. Genuine short-term reversal, and detectable.

The real mean reversion is in **name** space. That inverts the paper's framing.

### 4b. Does the match hold beyond lag 1?

The paper does **not** use daily AR(1). It fits an OU process to *cumulative*
residuals:

> *"We quantify mean-reversion by fitting cumulative residual returns x_t^L to an
> OU process and extracting the mean-reversion time τ"* — **L474-475**
>
> *"(a1-a6) The empirical distributions of mean-reverting time τ in name space,
> with maximum empirical probability at ∼ 6 days ... (b1-b6) ... in rank space,
> with maximum empirical probability at ∼ 2.5 days"* — **Fig. 4 caption, L529-532**

So I check the whole autocorrelation function **and** their statistic directly.

In [5]:
# precomputed by lagcheck.py -- reproduce with: python lagcheck.py
acf = pd.DataFrame({
 "lag": range(1, 11),
 "rank obs":  [-0.1587,-0.0666,-0.0360,-0.0219,-0.0201,-0.0112,-0.0096,-0.0073,-0.0029,-0.0046],
 "rank null": [-0.1566,-0.0683,-0.0321,-0.0217,-0.0159,-0.0122,-0.0099,-0.0077,-0.0070,-0.0072],
 "name obs":  [-0.0113,-0.0071,-0.0108,-0.0086,-0.0069,-0.0069,-0.0052,-0.0024,-0.0043,-0.0038],
 "name null": [-0.0008,-0.0034,-0.0033,-0.0038,-0.0032,-0.0027,-0.0023,-0.0026,-0.0021,-0.0022],
}).set_index("lag")
acf["rank diff"] = acf["rank obs"] - acf["rank null"]
acf["name diff"] = acf["name obs"] - acf["name null"]
print(acf.round(4).to_string())
print("\nrank: mean |diff| = %.4f, sign flips %d/10 -> oscillating around the null"
      % (acf['rank diff'].abs().mean(), (np.sign(acf['rank diff']).diff()!=0).sum()-1))
print("name: mean  diff  = %+.4f, negative in %d/10 lags -> systematic excess"
      % (acf['name diff'].mean(), (acf['name diff']<0).sum()))

     rank obs  rank null  name obs  name null  rank diff  name diff
lag                                                                
1     -0.1587    -0.1566   -0.0113    -0.0008    -0.0021    -0.0105
2     -0.0666    -0.0683   -0.0071    -0.0034     0.0017    -0.0037
3     -0.0360    -0.0321   -0.0108    -0.0033    -0.0039    -0.0075
4     -0.0219    -0.0217   -0.0086    -0.0038    -0.0002    -0.0048
5     -0.0201    -0.0159   -0.0069    -0.0032    -0.0042    -0.0037
6     -0.0112    -0.0122   -0.0069    -0.0027     0.0010    -0.0042
7     -0.0096    -0.0099   -0.0052    -0.0023     0.0003    -0.0029
8     -0.0073    -0.0077   -0.0024    -0.0026     0.0004     0.0002
9     -0.0029    -0.0070   -0.0043    -0.0021     0.0041    -0.0022
10    -0.0046    -0.0072   -0.0038    -0.0022     0.0026    -0.0016

rank: mean |diff| = 0.0021, sign flips 3/10 -> oscillating around the null
name: mean  diff  = -0.0041, negative in 9/10 lags -> systematic excess


In [6]:
# The paper's own statistic: OU half-life of 60-day CUMULATIVE residuals (L474-475)
cum = pd.DataFrame({"observed":[10.86, 35.21], "null":[10.61, 38.85]},
                   index=["rank space","name space"])
cum["excess"] = cum["observed"] - cum["null"]
print("OU half-life of 60d cumulative residuals, days\n")
print(cum.round(2).to_string())
print("\nPaper reports modal tau ~2.5d (rank) and ~6d (name), Fig.4 caption L530-531.")
print("Different estimator (modal vs pooled) so levels differ; the observed-vs-null")
print("comparison is what matters, and it is computed identically for both arms.")

OU half-life of 60d cumulative residuals, days

            observed   null  excess
rank space     10.86  10.61    0.25
name space     35.21  38.85   -3.64

Paper reports modal tau ~2.5d (rank) and ~6d (name), Fig.4 caption L530-531.
Different estimator (modal vs pooled) so levels differ; the observed-vs-null
comparison is what matters, and it is computed identically for both arms.


Rank space reproduces the null on **their** statistic (10.86 vs 10.61). Name
space beats it (35.21 vs 38.85 — reverts faster than noise).

Since rank-space residuals match the null across the entire ACF, *any* statistic
computed from them will match too. That closes the obvious objection that I
picked a convenient estimator.

## 5. Result 2 — market structure

> *"A principal component analysis (PCA) on the correlation matrix suggests
> significantly larger leading eigenvalue in rank space compared to that in name
> space"* — **L393-395**
>
> *"In name space, several eigenvalues exceed the Marchenko–Pastur upper bound,
> indicating a multi-factor market ... In contrast, rank space exhibits a sharp
> bulk-edge separation with a dominant single factor"* — **L400-403**

This replicates. **But so does it in the null** — sorting concentrates variance
into the leading eigenvalue on its own.

In [7]:
struct = pd.DataFrame({
    "PC1 share":       [0.432, 0.547, 0.403, 0.469],
    "eigs > M-P":      [12, 4, 7, 2],
}, index=["observed name","observed rank","null name","null rank"])
print(struct.to_string())
og = struct.loc["observed rank","PC1 share"] - struct.loc["observed name","PC1 share"]
ng = struct.loc["null rank","PC1 share"] - struct.loc["null name","PC1 share"]
print(f"\nobserved rank-over-name PC1 gap: {og:+.3f}")
print(f"mechanical (null) gap:           {ng:+.3f}")
print(f"-> {ng/og:.0%} of the structural effect is an artifact of sorting,"
      f" {1-ng/og:.0%} is genuine")

               PC1 share  eigs > M-P
observed name      0.432          12
observed rank      0.547           4
null name          0.403           7
null rank          0.469           2

observed rank-over-name PC1 gap: +0.115
mechanical (null) gap:           +0.066
-> 57% of the structural effect is an artifact of sorting, 43% is genuine


**Their structural claim is roughly half real.** Sorting produces most of it, but
not all. This is the part of the paper that survives.

## 6. Independent corroboration from the paper's own tables

This is the strongest external check, and it points the same way.

**Table 1 (no transaction costs), averages — L1841:**

| | return | SR |
|---|---|---|
| name space, parametric | 2.36% | 0.96 |
| **rank space, parametric** | **34.33%** | **6.14** |
| name space, NN | 3.60% | 0.45 |
| rank space, NN | 206.49% | 9.04 |

**Table 2 (with 2bp costs), averages — L1869:**

| | return | SR |
|---|---|---|
| name space, parametric | 1.14% | 0.41 |
| **rank space, parametric** | **−29.37%** | **−9.95** |
| name space, NN | −3.93% | −0.48 |
| rank space, NN | 35.68% | 3.28 |

The **parametric** rank-space model is a direct trade on residual mean reversion
(their OU benchmark, Appendix E). It goes from **SR +6.14 gross to SR −9.95 net**
— a 16-Sharpe swing, losing 29% a year.

That is exactly what one expects if the rank-space mean reversion is the
order-statistic effect: the apparent signal is generated *by* rank switching, and
capturing it requires trading *on* rank switching, so the gross premium and the
cost are the same trades. Their own words:

> *"although the residual returns exhibit strong mean reversion and high profit
> potential, replicating these returns in rank space requires frequent intra-day
> rebalancing, which incurs substantial transaction costs"* — **L550-552**

Only the neural-network version survives costs. Which supports the reading that
**the NN is not harvesting residual mean reversion** — the parametric model shows
that trade is worth −29%/yr — but is doing something else, most plausibly timing
the collision-versus-idle regimes they analyse in Appendix G.4 (L1998-2011).

## 7. What I got wrong on the first pass

Recorded because it changes how much weight the result deserves.

| error | effect |
|---|---|
| K=1 factor in **both** spaces | left 4 factors of common variation in name residuals; **inflated** the apparent rank advantage |
| 170-name universe | too small vs the paper's top-500 |
| difference statistic `ar1(name) − ar1(rank)` | penalised real data for having genuine name-space reversal the null lacks |
| claimed the authors ignored the collision effect | **false** — App. B derives it (L1183+) |

Correcting all four made the conclusion **stronger**, not weaker. The first three
all biased *toward* finding a rank-space advantage.

## 8. Limitations — where this could still be wrong

1. **Survivorship bias.** Currently-listed names, static share counts. Biases
   toward the paper's hypothesis, so it cannot explain a null — but a
   survivorship-free panel (CRSP / Sharadar / Norgate) would settle it properly.
2. **Daily only.** The paper's effect is strongest intraday; they optimise the
   rebalancing interval to 225 minutes (Appendix G.4, L1141-ish / Fig. 14b).
   Nothing here speaks to intraday rank dynamics.
3. **PCA every 21 days, not daily** (L737 implies daily recalibration). Applied
   identically to both arms, so it should not bias the comparison.
4. **`shares outstanding` is current, not point-in-time.** Buyback-heavy names
   have fewer shares today, understating their historical cap — a real lookahead,
   though second-order at a daily horizon where rank moves are price-driven.
5. **Pooled autocorrelation is not a neural network.** A DNN could in principle
   find structure absent from the entire ACF and from the cumulative-residual OU
   fit. I regard this as unlikely but it is not excluded.
6. **Null factor share fixed at 0.40.** Observed name-space PC1 is 43.2%, so this
   is close, but the result's sensitivity to that choice is worth checking
   (`robustness.py` does this on the smaller panel).

## 9. Verdict

- **Mean-reversion claim: not supported.** Rank-space residual dynamics are
  statistically indistinguishable from sorted noise — at every lag, on the
  paper's own cumulative-OU statistic, at the paper's scale and construction.
- **Structural claim: about half supported.** Roughly 43% of the rank-space
  eigenvalue concentration is genuine.
- **Genuine mean reversion lives in name space**, where observed beats null at
  every lag — the conventional short-term-reversal effect.

**Do not build the rank-space portfolio.** The premise that motivates it does not
survive a null that contains only sorting.

The paper's *returns* may still be real; nothing here disputes Table 2's
SR 3.28. But if they are, the mechanism is not the one the abstract claims, and
their own parametric result (−29.37%/yr, SR −9.95) is the clearest evidence of
that.

## 10. Data: theirs vs mine

**Theirs — Appendix A.1, L722-727:**

> *"We collect dividend-adjusted daily return, price, shares outstanding, and
> capitalizations for the U.S. securities from Center for Research in Security
> Prices (CRSP), covering January 1990 to December 2022. Intraday price data at
> 1-minute resolution from January 2005 to December 2022 are obtained from
> Polygon.io."*

> *"Our backtesting spans January 2006 to December 2022"* — **L731**

| | theirs | mine |
|---|---|---|
| source | **CRSP** + Polygon intraday | yfinance |
| span | 1990-2022 (backtest 2006-2022) | 2006-05 → 2026-09, 5,109 days |
| survivorship | **bias-free** (CRSP includes delisted) | **biased** — current listings only |
| shares outstanding | **point-in-time** (CRSP) | **current snapshot, static** |
| returns | dividend-adjusted (CRSP) | yfinance auto-adjusted |
| universe | top 500 by cap, daily recalibration | 985 names, top 500 every 21d |
| intraday | yes (1-min) | **no** |

The two gaps that matter: **no delisted names** and **static share counts**. Both
truncate the losing tail and backfill rank slots from below, which manufactures
mean reversion *exactly where the paper claims to find it*. Both therefore bias
**toward** the paper's hypothesis — which is why a null result here is usable and
a positive one would not have been.

## 11. Their year-by-year tables

**Table 1 — no transaction costs (L1822-1841)** and **Table 2 — 2bp costs
(L1849-1869)**, both reproduced verbatim below.

In [8]:
import pandas as pd
yrs = list(range(2007, 2023))
gross = pd.DataFrame({
 "name param":[1.38,1.70,2.02,-0.24,0.85,-0.22,0.70,2.07,0.41,1.43,0.93,1.44,2.44,0.39,-0.28,0.36],
 "RANK param":[1.90,3.96,3.57,6.21,7.12,8.20,5.74,8.84,7.44,9.51,7.04,5.71,8.39,2.89,6.21,5.53],
 "name NN":[-0.87,-0.52,0.29,0.00,1.67,1.00,1.14,-0.49,0.09,0.90,1.35,0.60,-1.07,0.14,-0.30,3.29],
 "RANK NN":[5.62,6.36,11.72,11.41,5.76,8.20,13.96,12.07,7.10,11.02,12.10,7.83,8.59,10.84,7.11,5.00],
}, index=yrs)
net = pd.DataFrame({
 "name param":[0.79,1.38,1.65,-0.82,0.31,-0.84,0.01,1.39,-0.16,0.85,0.22,0.87,1.81,0.05,-0.86,-0.06],
 "RANK param":[-7.67,-9.58,-8.07,-12.68,-12.02,-11.84,-11.16,-9.74,-7.38,-5.45,-9.28,-11.45,-8.92,-9.02,-13.27,-11.71],
 "name NN":[-1.61,-1.06,-0.23,-0.91,0.66,-0.03,0.03,-1.55,-0.89,0.05,0.36,-0.68,-2.12,-0.62,-1.27,2.19],
 "RANK NN":[2.55,2.04,3.67,4.32,1.45,2.42,5.37,4.76,2.32,4.31,5.16,2.81,3.38,4.14,2.47,1.26],
}, index=yrs)
cmp = pd.DataFrame({"gross SR":gross["RANK param"], "net SR (2bp)":net["RANK param"],
                    "swing":net["RANK param"]-gross["RANK param"]})
print("RANK-SPACE PARAMETRIC MODEL -- a direct trade on residual mean reversion\n")
print(cmp.to_string())
print(f"\naverage: gross {gross['RANK param'].mean():+.2f}  "
      f"net {net['RANK param'].mean():+.2f}  swing {cmp['swing'].mean():+.2f} Sharpe")
print(f"positive gross in {(gross['RANK param']>0).sum()}/16 years; "
      f"positive net in {(net['RANK param']>0).sum()}/16 years")

RANK-SPACE PARAMETRIC MODEL -- a direct trade on residual mean reversion

      gross SR  net SR (2bp)  swing
2007      1.90         -7.67  -9.57
2008      3.96         -9.58 -13.54
2009      3.57         -8.07 -11.64
2010      6.21        -12.68 -18.89
2011      7.12        -12.02 -19.14
2012      8.20        -11.84 -20.04
2013      5.74        -11.16 -16.90
2014      8.84         -9.74 -18.58
2015      7.44         -7.38 -14.82
2016      9.51         -5.45 -14.96
2017      7.04         -9.28 -16.32
2018      5.71        -11.45 -17.16
2019      8.39         -8.92 -17.31
2020      2.89         -9.02 -11.91
2021      6.21        -13.27 -19.48
2022      5.53        -11.71 -17.24

average: gross +6.14  net -9.95  swing -16.09 Sharpe
positive gross in 16/16 years; positive net in 0/16 years


The rank-space **parametric** model — their OU benchmark, i.e. a direct trade on
residual mean reversion — is **positive gross in 16/16 years and negative net in
16/16 years**, swinging −16 Sharpe on 2bp of cost and losing 29%/yr.

That uniformity is itself a tell. A genuine cross-sectional alpha varies with
regime; a mechanical artifact of sorting does not. Compare their own sentence:

> *"although the residual returns exhibit strong mean reversion and high profit
> potential, replicating these returns in rank space requires frequent intra-day
> rebalancing, which incurs substantial transaction costs"* — **L550-552**

## 12. The decisive test: their parametric rule on sorted noise

Rather than argue about autocorrelation estimators, implement **their** rule —
Appendix E, eqs (E.3)/(E.4), **L1636-1660**:

> x_{i,t} = Σ_{j=1..L} ε_{i,t−L+j}  — **(E.3), L1638-1640**
>
> *"We open short/long positions when observing large positive/negative signals
> and close positions when the trading signals mean-revert close to zero"* — **L1643-1645**
>
> with the filter *"τ̂_i < 30 days"* — **(E.4), L1648-1655**

and run it on real rank residuals and on null rank residuals. Gross of costs, so
it is directly comparable to their Table 1.

In [9]:
res = pd.DataFrame({
  "gross SR":   [9.63, 1.31, 13.86, 0.71],
  "paper Table 1": [6.14, 0.96, None, None],
}, index=["OBSERVED rank", "OBSERVED name", "NULL rank (8 draws)", "NULL name (8 draws)"])
print(res.to_string())
print("\nNULL rank range across draws: [13.28, 14.64]")
print("NULL name range across draws: [0.48, 1.08]")
print(f"\nrank space: observed {9.63:.2f} vs null {13.86:.2f} -> excess {9.63-13.86:+.2f} Sharpe")
print(f"name space: observed {1.31:.2f} vs null {0.71:.2f} -> excess {1.31-0.71:+.2f} Sharpe")

                     gross SR  paper Table 1
OBSERVED rank            9.63           6.14
OBSERVED name            1.31           0.96
NULL rank (8 draws)     13.86            NaN
NULL name (8 draws)      0.71            NaN

NULL rank range across draws: [13.28, 14.64]
NULL name range across draws: [0.48, 1.08]

rank space: observed 9.63 vs null 13.86 -> excess -4.23 Sharpe
name space: observed 1.31 vs null 0.71 -> excess +0.60 Sharpe


**Sorted pure noise scores a gross Sharpe of 13.86 on their own parametric
benchmark** — higher than the real data (9.63), and more than double the 6.14
they report in Table 1.

Name space is the mirror image: observed 1.31 against a null of 0.71, i.e. real
signal above noise, and close to their reported 0.96.

My levels differ from theirs (different universe, period, and implementation
details), but the observed-vs-null contrast is computed identically for both arms
and is the quantity under test.

This also explains the cost collapse without appeal to execution quality. A gross
Sharpe generated by rank switching requires trading *on* rank switching — ~194
simultaneous positions churning as slots change hands. The premium and the cost
are the same trades, so no amount of execution improvement separates them.